# Project description

Strategic Marketing Analytics: Customer Lifecycle & ROI Optimization

📌 Project Executive Summary
This analysis evaluates over 350,000 user sessions and transaction logs to optimize marketing efficiency for a digital platform. The primary objective is to move beyond surface-level traffic metrics and perform a deep dive into Unit Economics—specifically measuring how much it costs to acquire a customer (CAC) versus how much value they bring over time (LTV).

By identifying underperforming ad sources and analyzing cohort-based retention, this project provides a data-driven roadmap for reallocating marketing budgets to maximize the Return on Marketing Investment (ROMI).

🎯 Strategic Objectives

Engagement Analysis: Quantify user behavior by measuring Daily, Weekly, and Monthly Active Users (DAU/WAU/MAU) to identify product "stickiness".

Conversion Optimization: Analyze the time-to-conversion for different user segments to understand the typical buyer journey.

Revenue Modeling: Calculate Lifetime Value (LTV) through monthly cohort analysis to predict future revenue streams.

Marketing Efficiency Audit: Audit marketing spend across various sources to calculate CAC and ROI, identifying channels for budget expansion or phase-out.

🛠 Tech Stack & Methodology

Python (Pandas, NumPy): Used for large-scale data cleaning, timestamp optimization, and cohort aggregation.

Financial Visualization: Seaborn and Matplotlib leveraged for advanced business reporting, including retention heatmaps and ROMI growth curves.

Cohort Analysis: Segmented users by their first-visit month to track behavioral changes and profitability over a 12-month lifecycle

Contents
1. [Initial examination](#step_1)  
    a. Visits  
    b. Orders  
    c. Costs  
 
 
2. [Preproccesing](#step_2)   
    a. Visits  
    b. Orders  
    c. Costs  


3. [EDA - Product](#step_3)   
     a. How many people use it every day, week, and month?    
     b. How many sessions are there per day? (One user might have more than one session.)  
     c. What is the length of each session?
     d. How often do users come back?
  
  
4. [EDA - Sales](#step_4)   
     a. When do people start buying?  
     b. How many orders do they make during a given period of time?  
     c. What is the average purchase size?  
     d. How much money do they bring? (LTV)   
 
 
5. [EDA - Marketing](#step_5)  
     a. How much money was spent? Overall/per source/over time  
     b. How much did customer acquisition from each of the sources cost?  
     c. How worthwhile where the investments? (ROI)  
  
  
6. [Final Conclusions](#step_8)
    
</br>

 ## Initial examination <a id='step_1'></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats as st

In [ ]:
visits = pd.read_csv('data/visits_log_us.csv')
orders = pd.read_csv('data/orders_log_us.csv')
costs = pd.read_csv('data/costs_us.csv')

----------

### Visits

In [ ]:
visits.info()

In [ ]:
visits.head()

The visits table (server logs with data on website visits):

Uid — user's unique identifier  
Device — user's device  
Start Ts — session start date and time  
End Ts — session end date and time  
Source Id — identifier of the ad source the user came from  

All dates in this table are in YYYY-MM-DD format.   

<b>Conclusions:</b>  
So here we have all the timestamped visits (start and end) to the site with their ad source, user, and type of device. These columns can all be switched to a different type to decrease size. With 359400 rows it is the largest of the dataframes.


---------------

### Orders

In [ ]:
orders.info()

In [ ]:
orders.head()

The orders table (data on orders):

Uid — unique identifier of the user making an order  
Buy Ts — order date and time  
Revenue — Yandex.Afisha's revenue from the order  

<b>Conclusions:</b>  
This dataframe only has each order's timestamp, the revenue, and the id of the user who made the purchase. Their types will be switched to decrease size, and later I'll use this revenue to calculate a lot of the data per user.

----------

### Costs

In [ ]:
costs.info()

In [ ]:
costs.head()

The costs table (data on marketing expenses):

source_id — ad source identifier  
dt — date  
costs — expenses on this ad source on this day  

<b>Conclusions:</b>  
Finally we have our marketing expenses dataframe, including their ad source, timestamp, and cost. It's small comparatively with only 2542 rows, I'll be able to switch type to decrease the size of the dataframe, and I'll probably end up using the data to calculate monthly totals and adding them to my final dataframe.

-------------

 ## Preprocessing <a id='step_2'></a>

### Visits

In [ ]:
visits.isnull().sum()

In [ ]:
visits.info()

In [ ]:
visits.head(2)

In [ ]:
visits.rename(columns={'Device':'device', 'End Ts':'end_ts', 'Source Id':'source_id', 'Start Ts':'start_ts', 'Uid':'uid'}, inplace=True)
visits.columns

In [ ]:
print(visits['device'].unique())
visits['source_id'].unique()

In [ ]:
visits['device'] = visits['device'].astype('category')
visits['source_id'] = visits['source_id'].astype('int8')

visits['end_ts'] = pd.to_datetime(visits['end_ts'], format='%Y-%m-%d %H:%M:%S')
visits['start_ts'] = pd.to_datetime(visits['start_ts'], format='%Y-%m-%d %H:%M:%S')

visits.info()

--------

### Orders

In [ ]:
orders.isnull().sum()

In [ ]:
orders.info()

In [ ]:
orders.head(2)

In [ ]:
orders.rename(columns={'Buy Ts':'buy_ts', 'Revenue':'revenue', 'Uid':'uid'}, inplace=True)
orders.columns

In [ ]:
orders['revenue'].describe()

In [ ]:
orders['revenue'] = orders['revenue'].astype('float16')

orders['buy_ts'] = pd.to_datetime(orders['buy_ts'], format='%Y-%m-%d %H:%M:%S')
orders.info()

------------

### Costs

In [ ]:
costs.isnull().sum()

In [ ]:
costs.info()

In [ ]:
costs.head(2)

In [ ]:
costs['source_id'].unique()

In [ ]:
costs['costs'].describe()

In [ ]:
costs['source_id'] = costs['source_id'].astype('int8')
costs['costs'] = costs['costs'].astype('float16')

costs['dt'] = pd.to_datetime(costs['dt'], format='%Y-%m-%d')
costs.info()

<b> Preprocessing conclusions:  </b> 

Every dataframe was free of missing values, and I was able to drop their sizes considerably by converting:  
    strings -> datetime/category  
    large int/float -> smaller int/float  

This decrease in size speeds up processing. I also renamed the columns for programming ease and simplicity, making them lowercase and removing spaces in favor of underscores.

------------

## EDA - Product <a id='step_3'></a>

### How many people use it every day, week, and month?


#### Day

In [ ]:
visits.head(3)

In [ ]:
visits['date'] = pd.DatetimeIndex(visits['start_ts']).date
visits['week'] = pd.to_datetime(visits['start_ts']).dt.to_period('W')
visits['month_year'] = pd.to_datetime(visits['start_ts']).dt.to_period('M')
visits['year'] = pd.DatetimeIndex(visits['start_ts']).year

visits.head(3)

In [ ]:
date_visits_unique = visits.pivot_table(values='uid', index=['date'], aggfunc=lambda x: len(x.unique()))
date_visits_unique.head(5)

In [ ]:
date_visits_unique.plot.line(figsize=(15,7), legend=False)

plt.title('Daily Unique User Count', fontsize=14)
plt.xlabel('Date')
plt.ylabel('User Count')
plt.show()
plt.show()

<b>Graph conclusions:  </b>

Here we can see the rise and fall of daily unique users. The number was growing higher from mid-2017 to late 2017 but then started to falter. Overall There seems to be decline since it's peak in December 2017. That dramatic drop can perhaps be explained by a server crash or scheduled maintenance. The peak in November I will investigate further.


In [ ]:
date_visits_unique.reset_index(inplace=True)

print('Daily minimum:', date_visits_unique['uid'].min())
print('Daily max:', date_visits_unique['uid'].max())
print('Daily mean: {:.1f}'.format(date_visits_unique['uid'].mean()))
print('Daily median: {:.1f}'.format(date_visits_unique['uid'].median()))
print('Daily standard deviation: {:.1f}'.format(date_visits_unique['uid'].std()))
date_visits_unique['uid'].hist(bins=50, label='2017', figsize=[8,4])

plt.title('Daily Unique Visits')
plt.xlabel('Daily Unique Visits')
plt.ylabel('Count')
plt.show()

<b> Data and graph conclusions:  </b>

Here we can see we've had a minimum of one, to a max of 3,319. The average is 908 but the median is 921. In the histogram we can see that most days hover between 400 - 1400 unique users, a wide spread.


In [ ]:
date_visits_unique[date_visits_unique['uid'] > 3000]

Eureka! It turns out that 11/24/2017 was Black Friday, thus accounting for the surge in visits. I'm betting the sale prices were also very dramatic to match to increase visits thusly.

----------

#### Week

In [ ]:
week_visits_unique = visits.pivot_table(values='uid', index=['week'], aggfunc=lambda x: len(x.unique()))
week_visits_unique.plot.line(figsize=(12,5), legend=False)

plt.title('Weekly Unique User Count', fontsize=14)
plt.xlabel('Week')
plt.ylabel('Unique Visits')
plt.show()

In [ ]:
print('Weekly minimum:', week_visits_unique['uid'].min())
print('Weekly max:', week_visits_unique['uid'].max())
print('Weekly mean: {:.1f}'.format(week_visits_unique['uid'].mean()))
print('Weekly median: {:.1f}'.format(week_visits_unique['uid'].median()))
print('Weekly standard deviation: {:.1f}'.format(week_visits_unique['uid'].std()))

<b>Conclusions:</b>
Here we see a similar story to the daily users in overall trend. We had steady increase through the end of 2017, but then gradual decrease through 2018. However on a weekly level we see that we have even larger deviation, with a standard deviation of 2,049. Max weeks topped out at 10,586, and bottomed out at 2,021. The mean and median were very close, at around 5,700, implying low skewedness. 

----------

#### Month

In [ ]:
month_visits_unique = visits.pivot_table(values='uid', index=['month_year'], aggfunc=lambda x: len(x.unique()))
month_visits_unique.plot.line(figsize=(15,7), legend=False)
plt.title('Monthly Unique User Count', fontsize=14)

plt.title('Monthly Unique Visits')
plt.xlabel('Date')
plt.ylabel('Unique Visits')
plt.show()

month_visits_unique.reset_index(inplace=True)
print('Monthly minimum:', month_visits_unique['uid'].min())
print('Monthly max:', month_visits_unique['uid'].max())
print('Monthly mean: {:.1f}'.format(month_visits_unique['uid'].mean()))
print('Monthly median: {:.1f}'.format(month_visits_unique['uid'].median()))
print('Monthly standard deviation: {:.1f}'.format(month_visits_unique['uid'].std()))

<b> Conclusions: </b> 
The monthly graph shows a much more dramatic view of the global trend, rise through 2017 and fall through 2018. THe worst month was August 2017 with 11,631, while the best was November 32,797 - logical as it had the all-star Black Friday. The deviation from month to month is even larger here, with 7,546. The mean and median are again close, between about 23 and 24,000. 

#### Aggregated user metrics

In [ ]:
date_visits_unique.plot.line(figsize=(10,2), legend=False)
plt.title('Daily Unique Visits')
plt.xlabel('Date')
plt.ylabel('Count')

week_visits_unique.plot.line(figsize=(10,2), legend=False)
plt.title('Weekly Unique Visits')
plt.xlabel('Date')
plt.ylabel('Count')

month_visits_unique.plot.line(figsize=(10,2), legend=False)
plt.title('Monthly Unique Visits')
plt.xlabel('Date')
plt.ylabel('Count')

plt.show()

<b>Aggregated Conclusions:</b>
The DAU gives us a good picture of the variation between days, the WAU does the same for weeks, while the MAU shows best the overall trend - rising in 2017 and falling through 2018.


------------

### How many sessions are there per day? (One user might have more than one session.)


In [ ]:
sessions_per_user = visits.groupby(['date']).agg({'uid': ['count','nunique']})
sessions_per_user.columns = ['n_sessions', 'n_users']
sessions_per_user['sessions_per_user'] = sessions_per_user['n_sessions'] / sessions_per_user['n_users']
sessions_per_user.head()

In [ ]:
sessions_per_user['sessions_per_user'].plot.line(figsize=(14,4), legend=False)
plt.title('Sessions per Day')
plt.xlabel('Date')
plt.ylabel('Sessions')
plt.show()

print('Average sessions per day:', sessions_per_user['sessions_per_user'].mean())
print('Median sessions per day:',sessions_per_user['sessions_per_user'].median())

sessions_per_user['sessions_per_user'].plot.box(figsize=(14,4), legend=False)
plt.title('Distribution of Sessions Per Day')
plt.ylabel('Count')
plt.show()

<b>Conclusions:</b>
Between these two graphs and the average and medians, you can see that sessions per day doesn't really deviate from 1.



-------------

### What is the length of each session?

In [ ]:
visits['session_duration_sec'] = (visits['end_ts'] - visits['start_ts']).dt.seconds

In [ ]:
visits['session_duration_sec'].plot.box() 
plt.title('Session Duration Distribution', fontsize=14)
plt.ylabel('Count')
plt.show()

visits.query('session_duration_sec < 8000')['session_duration_sec'].plot.box() 
plt.title('Session Duration Distribution', fontsize=14)
plt.ylabel('Count')
plt.show()

print('Median session duration in seconds:', visits['session_duration_sec'].median())
print('Average session duration in seconds:', visits['session_duration_sec'].mean())
print('Mode of session duration in seconds:')
visits['session_duration_sec'].mode()

<b>Session duration conclusions:</b>
We can see in this data that most sessions hover between 0 and 1000 seconds. There are a lot of outliers, perhaps users switched tabs and left it up for ages. And with such short sessions of 0 seconds, one can assume that perhaps the user landed in error, or had reopened their browser only to navigate away asap. The average session is 643 seconds, and because the median is only 300 seconds we ca understand the skew formed by the outliers.
    

-----------

### How often do users come back?

In [ ]:
min_visit = visits.groupby(['uid'])['start_ts'].min().reset_index()
min_visit.columns = ['uid', 'first_visit']

retention_data = visits.merge(min_visit[['uid', 'first_visit']], 
                              on=['uid'], 
                              how='left')
retention_data.head()

In [ ]:
#adding cohorts
retention_data['cohort'] = retention_data['first_visit'].apply(lambda x: x.strftime('%Y-%m'))

#adding each row's user's retention
retention_data['retention_date'] = (
    (retention_data['start_ts'] - retention_data['first_visit']) / np.timedelta64(1, 'D')).astype(int)

#excluding visits that are the first visit, because then they are not an example of retention
retention_data_clean = retention_data[retention_data.start_ts != retention_data.first_visit]

retention_data_clean.head()

In [ ]:
retention_list = [0, 1, 2, 7, 14, 30]
def retention(retention_day, group_by):
    grouped = (retention_data_clean
               .query('retention_date==@retention_day')
               .groupby(group_by)['uid']
               .nunique()/retention_data_clean.groupby(group_by)['uid'].nunique())*100
    grouped = grouped.reset_index().rename(columns={'uid':'Retention_{}%'.format(retention_day)})
    return grouped

new_retention = []
for i in retention_list:
    new_retention.append(retention(i, 'cohort'))

new_retention = pd.concat(new_retention, axis=1)
new_retention = new_retention.loc[:, ~new_retention.columns.duplicated()]
new_retention.set_index('cohort', inplace=True)
new_retention

In [ ]:
new_retention.info()

In [ ]:
plt.figure(figsize=(10, 8))
plt.title('Retention Percent, by Cohort')
sns.heatmap(
    new_retention,
    annot=True,
    fmt='.1f',
    linewidths=1,
    linecolor='gray',
) 
plt.xlabel('Retention Rate')
plt.ylabel('Cohort First Session Month')
plt.show()

<b>Conclusions:</b>
Well it looks like long-term retention at the 30% mark for any cohort is rather low, at around 1%. However in the 14% bracket we see the strongest performance with the Black Friday cohort of 11-17. As for the earlier brackets, every cohort following that November almost did increasingly better than the last in the 0%, 1%, and 2% retention marks.

---------

##  EDA - Sales <a id='step_4'></a>

### When do people start buying? 
    (In KPI analysis, we're usually interested in knowing the time that elapses between registration and conversion — when the user becomes a customer. For example, if registration and the first purchase occur on the same day, the user might fall into category Conversion 0d. If the first purchase happens the next day, it will be Conversion 1d. You can use any approach that lets you compare the conversions of different cohorts, so that you can determine which cohort, or marketing channel, is most effective.)
   

In [ ]:
orders.head(3)

In [ ]:
visits.head(3)

In [ ]:
first_session = visits.groupby('uid')['date'].min()
first_session.name = 'first_session_date'
first_session.head()

In [ ]:
first_order = orders.groupby('uid')['buy_ts'].min()
first_order.name = 'first_order_date'
first_order.head()

In [ ]:
orders = orders.join(first_order,on='uid')
orders = orders.join(first_session,on='uid')

orders.head()

In [ ]:
orders['first_order_date'] = pd.DatetimeIndex(orders['first_order_date']).date
orders['days_to_conversion'] = orders['first_order_date'] - orders['first_session_date']
orders.head()

In [ ]:
orders['days_to_conversion'] = pd.to_numeric(orders['days_to_conversion'].dt.days, downcast='integer')
orders.head(3)

In [ ]:
def conversion_cohort(days):
    if days == 0:
        return '0d'
    if days >= 1 and days < 8:
        return '1-7d'
    if days > 7 and days < 31:
        return 'in 30d'
    if days > 30:
        return 'more than 30d'
    else:
        return 'error'

orders['conversion_bracket'] = orders['days_to_conversion'].apply(conversion_cohort)



orders['conversion_bracket'].value_counts().plot.pie(figsize=(5,5))
plt.title('Time of Conversion')
plt.show()

In [ ]:
orders['first_session_month'] = pd.to_datetime(orders['first_session_date']).dt.to_period('M')

cohort_conversion = orders.pivot_table(index='first_session_month', values='uid', columns='conversion_bracket', aggfunc='nunique')
cohort_conversion

In [ ]:
plt.figure(figsize=(10, 8))
plt.title('Number of Users Converted by Time Elapsed and Cohort')
sns.heatmap(
    cohort_conversion,
    annot=True,
    fmt='.1f',
    linewidths=1,
    linecolor='gray',
) 
plt.show()

<b>Conclusions:</b>
October, November, and December had the best numbers of users converted the first day, and continued to have the highest numbers in the first week and month. Finally there was a lot of conversions much later in their lifespan for all cohorts, with the June, October, and November 2017 cohorts having the highest conversion post 31 days. To be fair, 31-365 is a HUGE window, sweeping up a lot of slow-to-commit users.

----------

### How many orders do they make during a given period of time?

In [ ]:
orders['buy_ts_month'] = pd.to_datetime(orders['buy_ts']).dt.to_period('M')
orders.head()

In [ ]:
cohort_monthly_purchases = orders.pivot_table(
    index='first_session_month',
    columns='buy_ts_month',
    values='uid',
    aggfunc='count')

plt.figure(figsize=(10, 8))
plt.title('Number of Purchases')
sns.heatmap(
    cohort_monthly_purchases,
    annot=True,
    fmt='.1f',
    linewidths=1,
    linecolor='gray',
) 
plt.xlabel('Month of Purchases')
plt.ylabel('Month of First Session')
plt.show()

<b>Conclusions:</b>
October, November, and December 2017 are winners once again with the strongest showing in their first month. THe following January through March stays strong but them drops off by a third in April. No cohort makes as significant of a showing as they do in the first month, the drop is steep, and then steadily declines bit by bit each month. The magic mostly happens that first month.

-------------

### What is the average purchase size?

In [ ]:
orders_grouped_by_cohorts = orders.groupby(['first_session_month', 'buy_ts_month']).agg({'revenue': 'sum', 'uid': 'nunique'})
orders_grouped_by_cohorts.head()

In [ ]:
orders_grouped_by_cohorts['revenue_per_user'] = (orders_grouped_by_cohorts['revenue'] / orders_grouped_by_cohorts['uid']) 
orders_grouped_by_cohorts.head(3)

In [ ]:
cohort_avg_revenue = orders_grouped_by_cohorts.pivot_table(
    index='first_session_month',
    columns='buy_ts_month',
    values='revenue_per_user',
    aggfunc='mean') 
cohort_avg_revenue

In [ ]:
plt.figure(figsize=(10, 8))
plt.title('Avg Revenue Per Month, by Cohort')
sns.heatmap(
    cohort_avg_revenue,
    annot=True,
    fmt='.1f',
    linewidths=1,
    linecolor='gray',
) 
plt.xlabel('Month of Avg Revenue')
plt.ylabel('Cohort First Session Month')
plt.show()

<b>Conclusions:</b>
As you can see, regardless of cohort, most average revenue values stayed below 10. Three cohorts that were a bit higher were the first, 6-2017, then September 2017 and December 2017. Perhaps we had more than a few users buying in bulk?

### How much money do they bring? (LTV)

In [ ]:
orders_grouped_by_cohorts = orders_grouped_by_cohorts.reset_index()

orders_grouped_by_cohorts['cohort_lifetime'] = (orders_grouped_by_cohorts['buy_ts_month'].astype('int') - orders_grouped_by_cohorts['first_session_month'].astype('int'))
orders_grouped_by_cohorts['cohort_lifetime'] = orders_grouped_by_cohorts['cohort_lifetime'].astype('int8')
orders_grouped_by_cohorts.head()

In [ ]:
cohort_revenue = orders.groupby(['first_session_month','buy_ts_month']).agg({'revenue': 'sum'}).reset_index()

In [ ]:
orders.head()

In [ ]:
cohort_users = orders.pivot_table(values='uid', index='first_session_month', aggfunc='nunique')
cohort_users.drop('first_session_month', axis=1, inplace=True)
cohort_users.reset_index(inplace=True)

cohort_users

In [ ]:
report = pd.merge(cohort_users, cohort_revenue, on='first_session_month')
report.head(3) 

In [ ]:
margin_rate = .5

report['gp'] = report['revenue'] * margin_rate
report['age'] = (report['buy_ts_month'].astype('int') - report['first_session_month'].astype('int'))

report.rename(columns={"uid": "n_users"}, inplace=True)
report.head()

In [ ]:
report['ltv'] = report['gp'] / report['n_users']

output = report.pivot_table(
    index='first_session_month', 
    columns='age', 
    values='ltv', 
    aggfunc='sum')

output.fillna('') 

In [ ]:
output.cumsum(axis=1)

In [ ]:
cohort_sum_ltv = output.sum(axis=1).to_frame()
cohort_sum_ltv.reset_index(inplace=True)
cohort_sum_ltv.columns = ['cohort_start', 'total_ltv']
cohort_sum_ltv['total_ltv'] = cohort_sum_ltv['total_ltv'].round(2)
cohort_sum_ltv

In [ ]:
cohort_sum_ltv.plot.bar(x='cohort_start', y='total_ltv', figsize=(10,6), legend=False)
plt.title('Lifetime Value by Cohort')
plt.xlabel('Cohort Start Month')
plt.ylabel('Total LTV')
plt.show()

<b>Conclusions:</b>
The first cohort had the greatest lifetime value followed by September 2017 and then December 2017. 2018 had consistently lower values than 2017.

In [ ]:
# Get source for every user
visits_source = visits.sort_values('start_ts').groupby('uid')['source_id'].first().reset_index()
orders = orders.merge(visits_source, on='uid', how='left')
orders.head()

In [ ]:
source_revenue = orders.pivot_table(values='revenue', index='source_id', aggfunc='mean')
source_revenue.sort_values(by='revenue', ascending=False)

<b>Conclusions:</b> 
    Highest average revenue was for source 2, 1, 5 and 4 consecutively.  With source 7, then 10, and 9 doing the worst.

-----

## EDA - Marketing <a id='step_5'></a>

### How much money was spent? Overall/per source/over time

In [ ]:
costs.head(3)

In [ ]:
costs['purchase_month'] = pd.to_datetime(costs['dt']).dt.to_period('M')
cost_totals = costs.pivot_table(index='purchase_month', columns='source_id', values='costs', aggfunc='sum')
cost_totals

In [ ]:
cost_totals.plot.bar(stacked=True, figsize=(10,7))

plt.title('Costs per Source, by Month', fontsize=20)
plt.xlabel('Month', fontsize=15)
plt.ylabel('Monthly Total Cost', fontsize=15)
plt.show()

<b>Conclusions:</b>
The ad source that incurred the most cost was source 3, followed by similar numbers in 5 and 4, then 2, 1, and lastly 10 then 9. 


-----------

### How much did customer acquisition from each of the sources cost?

In [ ]:
# Get source for every user
visits_source = visits.sort_values('start_ts').groupby('uid')['source_id'].first().reset_index()

In [ ]:
orders.head()

In [ ]:
#This code now exists at line 72, because I used the sources to calculate something earlier
# Buyers amount for every first order date and source id
#orders_source = orders.merge(visits_source, on='uid', how='left')
#orders_source.head()

orders_source = orders

In [ ]:
orders_source['first_order_date'] = orders_source['first_order_date'].astype('datetime64[D]')
buyers_source = orders_source.groupby(['source_id', 'first_order_date']).agg({'uid':'nunique'}).\
                rename(columns={'uid':'n_buyers'}).reset_index()
buyers_source.head()

In [ ]:
# Create CAC table
cac_source_table = buyers_source.merge(costs, 
                                       left_on = ['source_id', 'first_order_date'],
                                       right_on = ['source_id', 'dt'])
cac_source_table['cac_source'] = cac_source_table['costs'] / cac_source_table['n_buyers']
cac_pivot_source = cac_source_table.pivot_table(index='purchase_month', 
                                                columns='source_id', 
                                                values='cac_source', 
                                                aggfunc='mean')

In [ ]:
cac_pivot_source

In [ ]:
plt.figure(figsize=(10, 8))
plt.title('Cost of Acquisiton, by Source and Cohort')
sns.heatmap(
    cac_pivot_source,
    annot=True,
    fmt='.1f',
    linewidths=1,
    linecolor='gray',
) 
plt.xlabel('Ad Source')
plt.ylabel('Cohort First Session Month')
plt.show()

<b>Conclusions:</b> 
Ad sources 2 and 3 bounce between having the highest CAC, followed by source 1. Sources 4, 9 and 10 bounce between having the lowest CAC. 

--------------------

### How worthwhile where the investments? (ROI)
    Plot graphs to display how these metrics differ for various devices and ad sources and how they change in time.

In [ ]:
####### LTV calculation

In [ ]:
# # CAC per month
# cac_month = cac_pivot_source.mean(axis=1).reset_index().rename(columns={0: 'cac'})

# # Update report with LTV
# report = report.merge(cac_month, left_on='first_session_month', right_on='purchase_month')

# # Calculate ROMI
# report['romi'] = report['ltv']/report['cac']

In [ ]:
# report

In [ ]:
# report.pivot_table(columns='first_session_month', index='age', values='romi', aggfunc='mean').cumsum(axis=1)

In [ ]:
########

In [ ]:
report.head()

In [ ]:
cac_pivot_source

In [ ]:
# calculating the costs per month
cac_month = cac_pivot_source.mean(axis=1).reset_index().rename(columns={0: 'cac'})
report = report.merge(cac_month, left_on='first_session_month', right_on='purchase_month')
report.head()

In [ ]:
#Calculate ROMI by dividing LTV by CAC.
report['romi'] = report['ltv'] / report['cac']
report

In [ ]:
report.pivot_table(columns='first_session_month', index='age', values='romi', aggfunc='mean').cumsum(axis=1).round(2)

In [ ]:
romi_output = report.pivot_table(index='first_session_month', columns='age', values='romi', aggfunc='mean').cumsum(axis=1).round(2)
romi_output

<b>Conclusions:</b>  
According to this graph no cohort has had a return on their investment.

-----------

## Final Conclusions <a id='step_6'></a>

#### Advise marketing experts how much money to invest and where. What sources/platforms would you recommend? Back up your choice: what metrics did you focus on? Why? What conclusions did you draw after finding the metric values? 

Earlier I examined LTV and CAC by cohort, so let's take a closer look at ad source for marketing:

In [ ]:
#Increasing size of floats so new calculations don't return infinity
orders['revenue'] = orders['revenue'].astype('float32')
costs['costs'] = costs['costs'].astype('float32')

#Totaling revenue by source
source_revenue_total = orders.pivot_table(values='revenue', index='source_id', aggfunc='sum').round()
source_revenue_total.reset_index(inplace=True)

#Totaling costs by source
source_costs_total = costs.pivot_table(index='source_id', values='costs', aggfunc='sum').round()
source_costs_total.reset_index(inplace=True)

In [ ]:
#Merging
source_compare = source_revenue_total.merge(source_costs_total, on='source_id')
source_compare.columns = ['source_id', 'total_revenue', 'total_marketing_costs']

#ROI of ad campaign = (income − expenses) / expenses
source_compare['ROI'] = (source_compare['total_revenue'] - source_compare['total_marketing_costs']) / source_compare['total_marketing_costs']
source_compare.sort_values(by='ROI', ascending=False)

<b>Conclusions:</b>
Upon close investigation of return on investment, the current business model as it stands is not sustainable as no group returned a profit after paying off their cost of acquisiton.

If the project has it's sights on potential future returns I would advise the marketing team to increase their investments in sources 1, 2, and 9. And furthermore to perhaps decrease spending in sources 3, 10, 4, and 5. In fact total marketing costs for source one are SO much lower than source 3, and in fact source 3 has the lowest ROI so I would recommend phasing out source 3 in favor of source 1. Indeed I would probably recommend they transfer the spending on the lower 4 in the chart above to the top 4, perhaps in phases to chart the effect - perhaps by narrowing the focus to just our top 4 sources by ROI we could increase our revenue while decreasing our costs.